# Bhutanese Code-Mixed Smart Home Assistant — Multitask DistilBERT

One multilingual DistilBERT encoder reads the command. Its shared representation feeds an intent head with nine classes and a device head with four classes. The device head learns from the `Device` column.

1. In Colab, select a GPU under **Runtime → Change runtime type** and choose **Run all**.
2. Authorize Google Drive and upload `dataset.xlsx` when prompted.
3. The notebook creates a timestamped folder under **My Drive → distilbert_multitask_runs** with the data splits, evaluation, and model export.

## 1. Set up the training environment

In [ ]:
%pip install -q "transformers==5.18.0" datasets accelerate scikit-learn openpyxl

In [ ]:
import json
from datetime import datetime
from io import BytesIO
from pathlib import Path
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.model_selection import GroupShuffleSplit
from transformers import AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments, set_seed

SEED = 42
MAX_LENGTH = 64
CONFIDENCE_THRESHOLD = 0.8
MODEL_NAME = "distilbert/distilbert-base-multilingual-cased"
print("Training device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Load and audit `dataset.xlsx`

The text input is `Transcript`. The targets are `Intent` and `Device`. `Language Type` is used to report English and code-mixed results. Rows without a transcript or either label are skipped.

In [ ]:
from google.colab import drive, files

drive.mount("/content/drive")
print("Upload dataset.xlsx.")
uploaded = files.upload()
if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith(".xlsx"):
    raise ValueError("Choose a single Excel workbook ending in .xlsx.")

data = pd.read_excel(BytesIO(next(iter(uploaded.values()))))
run_id = datetime.now(ZoneInfo("Asia/Thimphu")).strftime("%Y%m%d_%H%M%S")
run_dir = Path("/content/drive/MyDrive/distilbert_multitask_runs") / run_id
split_dir = run_dir / "splits"
model_dir = run_dir / "distilbert_multitask_model"
split_dir.mkdir(parents=True)
print("Run folder", run_dir)

In [ ]:
required_columns = {"Transcript", "Intent", "Device", "Language Type"}
missing = required_columns.difference(data.columns)
if missing:
    raise ValueError(f"Missing required workbook columns: {', '.join(sorted(missing))}.")

commands = data.dropna(subset=["Transcript", "Intent", "Device"]).copy()
commands["text"] = commands["Transcript"].map(lambda text: " ".join(str(text).split()))
commands = commands.loc[commands["text"].ne("")].copy()
for column in ("Intent", "Device"):
    commands[column] = commands[column].str.strip()

intent_names = ["Close", "DecreaseSpeed", "DecreaseTemp", "IncreaseSpeed", "IncreaseTemp", "Open", "SetTemp", "TurnOff", "TurnOn"]
device_names = ["AC", "Curtain", "Fan", "Light"]
intent_to_id = {name: index for index, name in enumerate(intent_names)}
device_to_id = {name: index for index, name in enumerate(device_names)}
for column, label_map in (("Intent", intent_to_id), ("Device", device_to_id)):
    if set(commands[column]) != set(label_map):
        raise ValueError(f"{column} labels must match the app labels {list(label_map)}.")

conflicting = commands.groupby("text")[["Intent", "Device"]].nunique().gt(1).any(axis=1)
if conflicting.any():
    examples = commands.loc[commands["text"].isin(conflicting[conflicting].index), ["text", "Intent", "Device"]]
    display(examples.drop_duplicates().head(20))
    raise ValueError("Resolve conflicting labels for identical transcripts before training.")

commands["label"] = commands["Intent"].map(intent_to_id)
commands["device_label"] = commands["Device"].map(device_to_id)
print(f"Using {len(commands):,} labeled commands from {len(data):,} workbook rows.")
display(pd.crosstab(commands["Intent"], commands["Device"]))

## 3. Create the train, validation, and test splits

Group by whitespace-normalized transcript and split approximately 70/15/15 with seed 42, matching the earlier intent notebook's split procedure. Check transcript overlap before training and save the assignments for future comparisons.

In [ ]:
train_indices, temp_indices = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED).split(
    commands, groups=commands["text"],
))
temporary = commands.iloc[temp_indices]
validation_indices, test_indices = next(GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED).split(
    temporary, groups=temporary["text"],
))
splits = {
    "train": commands.iloc[train_indices].reset_index(drop=True),
    "validation": temporary.iloc[validation_indices].reset_index(drop=True),
    "test": temporary.iloc[test_indices].reset_index(drop=True),
}

transcripts = {name: set(table["text"]) for name, table in splits.items()}
for first, second in (("train", "validation"), ("train", "test"), ("validation", "test")):
    overlap = transcripts[first] & transcripts[second]
    print(f"{first} / {second} transcript overlap: {len(overlap)}")
    assert not overlap

if set(splits["train"]["Intent"]) != set(intent_names) or set(splits["train"]["Device"]) != set(device_names):
    raise ValueError("The training split must contain each intent and device class.")

columns = ["text", "label", "device_label", "Intent", "Device", "Language Type"]
if "Control value" in commands:
    columns.append("Control value")
for name, table in splits.items():
    table[columns].to_csv(split_dir / f"{name}.csv", index=False)
    print(f"{name}: {len(table):,} commands")

## 4. Tokenize commands

The cased multilingual tokenizer maps each transcript to input IDs. The two class IDs are passed to the trainer as separate targets. Padding is applied per batch.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
dataset = DatasetDict({
    name: Dataset.from_pandas(table[["text", "label", "device_label"]], preserve_index=False)
    for name, table in splits.items()
})


def tokenize_batch(batch):
    inputs = tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, return_token_type_ids=False)
    return {**inputs, "intent_labels": batch["label"], "device_labels": batch["device_label"]}


encoded = dataset.map(tokenize_batch, batched=True, remove_columns=dataset["train"].column_names)
collator = DataCollatorWithPadding(tokenizer)
print("Tokens for the first training command", tokenizer.convert_ids_to_tokens(encoded["train"][0]["input_ids"]))

## 5. One encoder and two heads

Use the first token's final hidden state for both classifiers. Average the two cross-entropy losses and update the shared encoder and both heads together. The encoder starts from multilingual pretrained weights and the heads start with fresh weights.

In [ ]:
"""DistilBERT encoder with separate intent and device classification heads."""

from torch import nn
from torch.nn.functional import cross_entropy
from transformers import DistilBertConfig, DistilBertModel, DistilBertPreTrainedModel


class DistilBertForIntentAndDevice(DistilBertPreTrainedModel):
    config_class = DistilBertConfig

    def __init__(self, config):
        super().__init__(config)
        self.distilbert = DistilBertModel(config)
        self.dropout = nn.Dropout(config.seq_classif_dropout)
        self.intent_classifier = nn.Linear(config.dim, config.num_labels)
        self.device_classifier = nn.Linear(config.dim, len(config.device_id2label))
        self.post_init()

    def forward(self, input_ids, attention_mask=None, intent_labels=None, device_labels=None):
        encoded = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        command = self.dropout(encoded.last_hidden_state[:, 0])
        intent_logits = self.intent_classifier(command)
        device_logits = self.device_classifier(command)
        outputs = {"intent_logits": intent_logits, "device_logits": device_logits}
        if intent_labels is not None and device_labels is not None:
            intent_loss = cross_entropy(intent_logits, intent_labels)
            device_loss = cross_entropy(device_logits, device_labels)
            outputs["loss"] = (intent_loss + device_loss) / 2
        return outputs

In [ ]:
set_seed(SEED)
config = DistilBertConfig.from_pretrained(
    MODEL_NAME, id2label=dict(enumerate(intent_names)), label2id=intent_to_id,
)
config.device_id2label = {str(index): name for index, name in enumerate(device_names)}
config.command_max_length = MAX_LENGTH
model = DistilBertForIntentAndDevice.from_pretrained(MODEL_NAME, config=config, use_safetensors=True)

## 6. Validation metrics and training

Choose the checkpoint with the highest validation joint accuracy. A prediction counts as jointly correct when both intent and device match. Weighted F1 is reported for each head.

In [ ]:
def score_predictions(intent_actual, device_actual, intent_predicted, device_predicted):
    intent_correct = intent_predicted == intent_actual
    device_correct = device_predicted == device_actual
    return {
        "intent_accuracy": float(intent_correct.mean()),
        "device_accuracy": float(device_correct.mean()),
        "joint_accuracy": float((intent_correct & device_correct).mean()),
        "intent_f1_weighted": float(f1_score(intent_actual, intent_predicted, average="weighted", zero_division=0)),
        "device_f1_weighted": float(f1_score(device_actual, device_predicted, average="weighted", zero_division=0)),
    }


def compute_metrics(result):
    intent_logits, device_logits = result.predictions
    intent_actual, device_actual = result.label_ids
    return score_predictions(intent_actual, device_actual, intent_logits.argmax(axis=1), device_logits.argmax(axis=1))

In [ ]:
training_args = TrainingArguments(
    output_dir=str(run_dir / "checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="joint_accuracy",
    greater_is_better=True,
    save_total_limit=2,
    logging_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available(),
    label_names=["intent_labels", "device_labels"],
    seed=SEED,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["validation"],
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)
trainer.train()
print("Best checkpoint", trainer.state.best_model_checkpoint)
print("Best validation joint accuracy", trainer.state.best_metric)

## 7. Evaluate on the held-out test split

Report intent, device, and joint accuracy for all test commands and by language type. The app compares the lower of the two head scores with its 0.8 cutoff. The figures below show how many predictions have both labels correct before and after applying that cutoff.

In [ ]:
prediction = trainer.predict(encoded["test"])
intent_logits, device_logits = prediction.predictions
intent_actual, device_actual = prediction.label_ids
intent_predicted = intent_logits.argmax(axis=1)
device_predicted = device_logits.argmax(axis=1)
intent_score = torch.softmax(torch.from_numpy(intent_logits), dim=1).amax(dim=1).numpy()
device_score = torch.softmax(torch.from_numpy(device_logits), dim=1).amax(dim=1).numpy()
confidence = np.minimum(intent_score, device_score)
accepted = confidence >= CONFIDENCE_THRESHOLD
joint_correct = (intent_actual == intent_predicted) & (device_actual == device_predicted)

test = splits["test"]
test_metrics = {"All": score_predictions(intent_actual, device_actual, intent_predicted, device_predicted)}
for language in sorted(test["Language Type"].dropna().unique()):
    mask = test["Language Type"].eq(language).to_numpy()
    test_metrics[language] = score_predictions(
        intent_actual[mask], device_actual[mask], intent_predicted[mask], device_predicted[mask],
    )
display(pd.DataFrame.from_dict(test_metrics, orient="index").round(4))

confidence_metrics = {
    "threshold": CONFIDENCE_THRESHOLD,
    "accepted": int(accepted.sum()),
    "joint_correct_accepted": int((accepted & joint_correct).sum()),
    "joint_incorrect_accepted": int((accepted & ~joint_correct).sum()),
}
print("Confidence cutoff results", confidence_metrics)

In [ ]:
for title, actual, predicted, names in (
    ("Intent", intent_actual, intent_predicted, intent_names),
    ("Device", device_actual, device_predicted, device_names),
):
    print(title, "classification report")
    label_ids = list(range(len(names)))
    print(classification_report(actual, predicted, labels=label_ids, target_names=names, digits=4, zero_division=0))
    _, axis = plt.subplots(figsize=(9, 6))
    ConfusionMatrixDisplay.from_predictions(
        actual, predicted, labels=label_ids, display_labels=names,
        xticks_rotation=45, cmap="Blues", ax=axis,
    )
    axis.set_title(f"Test set — {title}")
    plt.tight_layout()
    plt.show()

In [ ]:
results = test[["text", "Language Type", "Intent", "Device"]].copy()
results["predicted_intent"] = [intent_names[index] for index in intent_predicted]
results["predicted_device"] = [device_names[index] for index in device_predicted]
results["confidence"] = confidence
results["accepted"] = accepted
results["both_labels_correct"] = joint_correct
results.to_csv(run_dir / "test_predictions.csv", index=False)
display(results.loc[~joint_correct].head(20))

## 8. Save and verify the app export

The export contains the shared encoder, both heads, label maps in `config.json`, and the tokenizer. Copy the listed files into `models/distilbert_multitask_model/` in the app and restart. The app continues to extract an explicit temperature value for `SetTemp` commands.

In [ ]:
trainer.save_model(str(model_dir))
tokenizer.save_pretrained(model_dir)
export_files = ("model.safetensors", "config.json", "tokenizer.json", "tokenizer_config.json")
for filename in export_files:
    path = model_dir / filename
    if not path.is_file():
        raise FileNotFoundError(f"Missing export file: {path}")
    print(f"{filename}: {path.stat().st_size / 1_000_000:.2f} MB")

metrics = {
    "base_model": MODEL_NAME,
    "samples": {name: len(table) for name, table in splits.items()},
    "best_validation_joint_accuracy": float(trainer.state.best_metric),
    "test_metrics": test_metrics,
    "confidence_cutoff": confidence_metrics,
}
(run_dir / "final_results.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("Download the four export files from", model_dir)

## 9. Demo the saved model

Try English, code-mixed, and paraphrased commands. Use the held-out test metrics above for the presentation. The sample below shows both predicted labels and the app's confidence score.

In [ ]:
saved_tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
saved_model = DistilBertForIntentAndDevice.from_pretrained(
    model_dir, local_files_only=True, use_safetensors=True,
).eval()
examples = [
    "fan on bay nang",
    "Set the AC to twenty-two degrees",
    "Let some daylight into the room",
    "Give the room some airflow",
    "Turn it on",
]

inputs = saved_tokenizer(
    examples, padding=True, truncation=True, max_length=saved_model.config.command_max_length,
    return_tensors="pt", return_token_type_ids=False,
)
with torch.inference_mode():
    output = saved_model(**inputs)
    intent_scores, intent_ids = output["intent_logits"].softmax(dim=1).max(dim=1)
    device_scores, device_ids = output["device_logits"].softmax(dim=1).max(dim=1)

scores = torch.minimum(intent_scores, device_scores).tolist()
display(pd.DataFrame({
    "Command": examples,
    "Intent": [saved_model.config.id2label[index] for index in intent_ids.tolist()],
    "Device": [saved_model.config.device_id2label[str(index)] for index in device_ids.tolist()],
    "Confidence": scores,
    "Passes confidence cutoff": [score >= CONFIDENCE_THRESHOLD for score in scores],
}))